# Bank Nifty Performance – Return Prediction
**Project ID 20 · ML with Big Data (UE24AM342AA4) · Author: Mahesh**

**Goal:** predict the **next-day return of the Nifty Bank index (`^NSEBANK`)** from information available at today's close, and compare five modelling frameworks:

| # | Framework | Role |
|---|-----------|------|
| 1 | **XGBoost** | gradient-boosted trees |
| 2 | **Keras** | feed-forward neural network |
| 3 | **PyTorch + fastai** | `tabular_learner` |
| 4 | **PyTorch + TabNet** | attention-based tabular DL |
| 5 | **PyTorch Lightning** | structured PyTorch training loop (Lightning Flash is deprecated, so plain Lightning is used) |

Each model is compared with simple baselines on **RMSE, MAE, R², directional accuracy**, an economic **back-test (Sharpe)**, and on **simplicity / transparency / efficacy**.

**Notebook outline:** 1 Setup → 2 Data → 3 EDA → 4 Feature engineering → 5 Split → 6 Baselines → 7–11 Models → 12 Walk-forward check → 13 Comparison → 14 Explainability → 15 Back-test → 16 Conclusion

## 1. Setup

In [ ]:
# Run this first in Google Colab / a fresh environment.
!pip -q install yfinance xgboost keras fastai pytorch-tabnet lightning scikit-learn seaborn scipy

import os, time, warnings, random
warnings.filterwarnings("ignore")
os.environ.setdefault("KERAS_BACKEND", "torch")   # use "tensorflow" if you prefer the TF backend

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
    except ImportError:
        pass
set_seed()

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 150)

## 2. Data
The brief asks us to pull history from **Yahoo Finance**. The cell below tries `yfinance` first (ticker `^NSEBANK`) and falls back to the provided `bank_nifty_dataset.csv` (same source, 17-Sep-2007 → 27-Oct-2026) if there is no internet or the download fails.

In [ ]:
CSV_PATH = "bank_nifty_dataset.csv"      # put the provided CSV next to the notebook (on Colab: upload it)
USE_YFINANCE = True

def load_from_yfinance():
    import yfinance as yf
    raw = yf.Ticker("^NSEBANK").history(start="2007-09-17", auto_adjust=False)
    raw.index = pd.to_datetime(raw.index).tz_localize(None).normalize()
    raw.index.name = "Date"
    return raw[["Open", "High", "Low", "Close", "Volume"]]

def load_from_csv(path):
    # file has 3 header rows (Price / Ticker / Date) exported by yfinance, dates are dd-mm-yyyy
    names = ["Date", "AdjClose", "Close", "High", "Low", "Open", "Volume"]
    d = pd.read_csv(path, skiprows=3, header=None, names=names)
    d["Date"] = pd.to_datetime(d["Date"], format="%d-%m-%Y")
    return d.set_index("Date")[["Open", "High", "Low", "Close", "Volume"]].astype(float)

df, source = None, None
if USE_YFINANCE:
    try:
        df = load_from_yfinance(); source = "yfinance (live)"
        assert len(df) > 1000
    except Exception as e:
        print("yfinance unavailable ->", type(e).__name__); df = None
if df is None:
    for p in [CSV_PATH, "/mnt/user-data/uploads/1793242595975_bank_nifty_dataset.csv", "/content/bank_nifty_dataset.csv"]:
        if os.path.exists(p):
            df = load_from_csv(p); source = f"CSV ({p})"; break
assert df is not None, "Could not load data - upload bank_nifty_dataset.csv"

df = df[~df.index.duplicated()].sort_index().dropna(subset=["Close"])
print("Source:", source)
print("Shape :", df.shape, "|", df.index.min().date(), "->", df.index.max().date())
df.head()

In [ ]:
display(df.describe().T)
print("Missing values:\n", df.isna().sum().to_string())
print("\nDays with volume == 0 : %.1f%%" % (100 * (df.Volume == 0).mean()))

**Data-quality notes.** The index is a single series (not per-stock rows as the brief's column list suggests), so there is no `symbol / stock / dividends`. Volume is **unreliable** for an index: it is 0 on a large share of days and has extreme spikes. It is kept only in a robust form (see feature engineering); price features carry the signal.

## 3. Exploratory Data Analysis

In [ ]:
df["ret"] = df["Close"].pct_change() * 100          # daily % return
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(df.index, df["Close"], lw=1); ax[0].set_yscale("log"); ax[0].set_title("Bank Nifty close (log scale)")
ax[1].plot(df.index, df["ret"], lw=.6, color="tab:red"); ax[1].set_title("Daily return (%)")
plt.tight_layout(); plt.show()

In [ ]:
r = df["ret"].dropna()
jb = stats.jarque_bera(r)
print(f"mean={r.mean():.3f}%  std={r.std():.3f}%  skew={stats.skew(r):.2f}  excess kurtosis={stats.kurtosis(r):.2f}")
print(f"Jarque-Bera p-value = {jb.pvalue:.2e}  ->  returns are NOT normal (fat tails)")
print(f"Up days: {(r>0).mean():.1%}   Best day: {r.max():.1f}%   Worst day: {r.min():.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(r, bins=120, kde=True, ax=ax[0]); ax[0].set_title("Return distribution"); ax[0].set_xlim(-8, 8)
stats.probplot(r, dist="norm", plot=ax[1]); ax[1].set_title("Q-Q plot vs normal")
plt.tight_layout(); plt.show()

In [ ]:
# Autocorrelation: returns vs squared returns (volatility clustering)
def acf(x, nlags=30):
    x = np.asarray(x) - np.mean(x)
    return np.array([np.sum(x[k:] * x[:len(x)-k]) / np.sum(x * x) for k in range(1, nlags + 1)])
lags = np.arange(1, 31); band = 1.96 / np.sqrt(len(r))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=False)
for a, series, t in zip(ax, [r, r**2], ["ACF of returns", "ACF of squared returns"]):
    a.bar(lags, acf(series)); a.axhline(band, ls="--", c="r"); a.axhline(-band, ls="--", c="r"); a.set_title(t); a.set_xlabel("lag")
plt.tight_layout(); plt.show()
print("Lag-1 autocorrelation of returns: %.3f  | of squared returns: %.3f" % (acf(r, 1)[0], acf(r**2, 1)[0]))

**Reading the plots:** returns have almost no autocorrelation (direction is hard to predict), while *squared* returns are strongly autocorrelated – **volatility clusters**. This tells us that *magnitude/risk* features (rolling volatility, ranges) should be informative, and that we should expect a low R² for next-day return.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
yr = df["ret"].groupby(df.index.year).agg(lambda x: ((1 + x/100).prod() - 1) * 100)
yr.plot.bar(ax=ax[0], color=np.where(yr >= 0, "tab:green", "tab:red")); ax[0].set_title("Calendar-year return (%)")
dow = df["ret"].groupby(df.index.dayofweek).mean(); dow.index = ["Mon","Tue","Wed","Thu","Fri"][:len(dow)]
dow.plot.bar(ax=ax[1]); ax[1].set_title("Mean return by weekday (%)")
vol = df["ret"].rolling(21).std() * np.sqrt(252)
ax[2].plot(vol); ax[2].set_title("21-day rolling annualised volatility (%)")
plt.tight_layout(); plt.show()

In [ ]:
# Volume sanity check
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
(df["Volume"] == 0).groupby(df.index.year).mean().plot.bar(ax=ax[0]); ax[0].set_title("Share of zero-volume days per year")
np.log1p(df["Volume"][df["Volume"] > 0]).plot.hist(bins=60, ax=ax[1]); ax[1].set_title("log(1+Volume), non-zero days")
plt.tight_layout(); plt.show()

## 4. Feature Engineering
**Target:** `target = return (%) on day t+1`.
**Features** use only data up to the close of day *t* (no look-ahead). Groups:
* **Momentum** – returns over 1, 2, 3, 5, 10, 21 days
* **Volatility** – rolling std (5/10/21/63d), ATR, high-low range, volatility-regime ratio *(innovation: captures clustering found in EDA)*
* **Trend / mean-reversion** – distance from SMA 5/10/20/50/200, MACD, RSI-14, Bollinger %B, drawdown from 252-day high
* **Intraday structure** – overnight gap, close location within the day's range
* **Calendar** – cyclical encoding of weekday and month
* **Volume (robust)** – relative volume vs 20-day mean, clipped; plus `has_volume` flag

In [ ]:
def make_features(d):
    c, o, h, l, v = d["Close"], d["Open"], d["High"], d["Low"], d["Volume"]
    r = c.pct_change() * 100
    X = pd.DataFrame(index=d.index)
    X["ret_1"] = r
    for k in [2, 3, 5, 10, 21]:
        X[f"ret_{k}"] = c.pct_change(k) * 100
    for w in [5, 10, 21, 63]:
        X[f"vol_{w}"] = r.rolling(w).std()
    X["vol_regime"] = X["vol_5"] / X["vol_63"]                       # short vs long vol
    for w in [5, 10, 20, 50, 200]:
        X[f"sma_dist_{w}"] = (c / c.rolling(w).mean() - 1) * 100
    # RSI-14 (Wilder)
    delta = c.diff(); up = delta.clip(lower=0); dn = -delta.clip(upper=0)
    rs = up.ewm(alpha=1/14, adjust=False).mean() / dn.ewm(alpha=1/14, adjust=False).mean()
    X["rsi_14"] = 100 - 100 / (1 + rs)
    # MACD histogram (normalised)
    macd = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
    X["macd_hist"] = (macd - macd.ewm(span=9, adjust=False).mean()) / c * 100
    # Bollinger %B
    m, s = c.rolling(20).mean(), c.rolling(20).std()
    X["bb_pctb"] = (c - (m - 2*s)) / (4*s)
    # ATR-14 % of price and daily range
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    X["atr_14"] = tr.rolling(14).mean() / c * 100
    X["hl_range"] = (h - l) / c * 100
    X["gap"] = (o / c.shift() - 1) * 100
    X["close_loc"] = (c - l) / (h - l).replace(0, np.nan)
    X["dd_252"] = (c / c.rolling(252, min_periods=60).max() - 1) * 100
    # calendar (cyclical)
    X["dow_sin"] = np.sin(2*np.pi*d.index.dayofweek/5); X["dow_cos"] = np.cos(2*np.pi*d.index.dayofweek/5)
    X["mon_sin"] = np.sin(2*np.pi*d.index.month/12);    X["mon_cos"] = np.cos(2*np.pi*d.index.month/12)
    # robust volume
    vm = v.replace(0, np.nan).rolling(20, min_periods=5).mean()
    X["vol_rel"] = (v.replace(0, np.nan) / vm).fillna(1).clip(0, 5)
    X["has_volume"] = (v > 0).astype(float)
    X["close_loc"] = X["close_loc"].fillna(0.5)
    X["target"] = r.shift(-1)                                         # NEXT-day return
    return X

data = make_features(df).replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=["target"])                   # last day has no label
feat_cols = [c for c in data.columns if c != "target"]
data = data.dropna()                                    # warm-up rows (200-day SMA etc.)
print("Rows after warm-up:", len(data), "| #features:", len(feat_cols))
data.head()

In [ ]:
corr = data.corr()["target"].drop("target").sort_values()
fig, ax = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1.2]})
corr.plot.barh(ax=ax[0]); ax[0].set_title("Correlation of each feature with next-day return")
sns.heatmap(data[feat_cols].corr(), cmap="coolwarm", center=0, ax=ax[1], cbar_kws={"shrink": .7}); ax[1].set_title("Feature correlation")
plt.tight_layout(); plt.show()
print("Max |corr| with target: %.3f  -> every single feature is a weak predictor (as expected for daily returns)" % corr.abs().max())

## 5. Train / Validation / Test split
Financial data is a time series, so we split **chronologically** (no shuffling): 70 % train, 15 % validation (early stopping / tuning), 15 % test (touched once for the final comparison). The scaler is fitted on the training part only.

In [ ]:
n = len(data); i_tr, i_va = int(n * .70), int(n * .85)
train, val, test = data.iloc[:i_tr], data.iloc[i_tr:i_va], data.iloc[i_va:]
for nm, s in [("train", train), ("val", val), ("test", test)]:
    print(f"{nm:5s}: {len(s):5d} rows  {s.index.min().date()} -> {s.index.max().date()}")

X_tr, y_tr = train[feat_cols].values, train["target"].values
X_va, y_va = val[feat_cols].values,   val["target"].values
X_te, y_te = test[feat_cols].values,  test["target"].values

scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_va, Xs_te = scaler.transform(X_tr), scaler.transform(X_va), scaler.transform(X_te)
Xs_tr, Xs_va, Xs_te = [a.astype("float32") for a in (Xs_tr, Xs_va, Xs_te)]
y_tr32, y_va32 = y_tr.astype("float32"), y_va.astype("float32")

## 6. Evaluation helpers & baselines
Metrics: **RMSE, MAE, R²** (on the test set), **directional accuracy** (did we get the sign right?), and a simple **long/short back-test** (position = sign of prediction; 0.02 % cost per position change) summarised by **Sharpe ratio**.

Baselines every model must beat: **zero-return** forecast, **train-mean** forecast, and **Ridge regression**.

In [ ]:
results, preds, train_time = {}, {}, {}
COST = 0.0002   # 2 bps per position change

def backtest(pred, actual_pct, cost=COST):
    pos = np.sign(pred)
    pos[pos == 0] = 0
    turn = np.abs(np.diff(np.r_[0, pos]))
    strat = pos * actual_pct / 100 - turn * cost
    return strat

def sharpe(x): return np.sqrt(252) * np.mean(x) / (np.std(x) + 1e-12)

def evaluate(name, pred, y=y_te, secs=0.0):
    pred = np.asarray(pred).ravel().astype(float)
    strat = backtest(pred.copy(), y)
    results[name] = dict(
        RMSE=np.sqrt(mean_squared_error(y, pred)), MAE=mean_absolute_error(y, pred), R2=r2_score(y, pred),
        DirAcc=np.mean(np.sign(pred) == np.sign(y)) * 100, Sharpe=sharpe(strat),
        AnnRet=(np.prod(1 + strat) ** (252 / len(strat)) - 1) * 100, Train_s=secs)
    preds[name] = pred
    return pd.Series(results[name]).round(4)

evaluate("Baseline: zero", np.zeros_like(y_te))
evaluate("Baseline: train mean", np.full_like(y_te, y_tr.mean()))
t = time.time(); ridge = Ridge(alpha=10).fit(Xs_tr, y_tr); evaluate("Baseline: Ridge", ridge.predict(Xs_te), secs=time.time() - t)
pd.DataFrame(results).T.round(4)

> **Note on R².** R² is measured against the *test-set mean*. For daily returns an honest model typically scores **≈ 0 or slightly negative**; the useful comparison is whether a model beats the zero/mean baselines on RMSE and whether any edge survives in the back-test.

## 7. Model 1 – XGBoost

In [ ]:
import xgboost as xgb
set_seed()
grid = [dict(max_depth=d, learning_rate=lr, min_child_weight=mcw)
        for d in (2, 3, 4) for lr in (0.02, 0.05) for mcw in (20, 50)]
best, best_rmse = None, 1e9
t0 = time.time()
for g in grid:
    m = xgb.XGBRegressor(n_estimators=1000, subsample=.8, colsample_bytree=.8, reg_lambda=5, early_stopping_rounds=50,
                         random_state=SEED, n_jobs=-1, **g)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    rm = m.best_score
    if rm < best_rmse: best, best_rmse, best_cfg = m, rm, g
xgb_time = time.time() - t0
print("Best params:", best_cfg, "| best iteration:", best.best_iteration, "| val RMSE: %.4f" % best_rmse)
evaluate("XGBoost", best.predict(X_te), secs=xgb_time)

## 8. Model 2 – Keras (feed-forward network)

In [ ]:
import keras
from keras import layers
set_seed(); keras.utils.set_random_seed(SEED)
def build_keras(n_in):
    return keras.Sequential([
        keras.Input(shape=(n_in,)),
        layers.Dense(64, activation="relu"), layers.Dropout(.3),
        layers.Dense(32, activation="relu"), layers.Dropout(.2),
        layers.Dense(1)])
k_model = build_keras(Xs_tr.shape[1])
k_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
t0 = time.time()
hist = k_model.fit(Xs_tr, y_tr32, validation_data=(Xs_va, y_va32), epochs=100, batch_size=64, verbose=0,
                   callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
keras_time = time.time() - t0
print("Epochs run:", len(hist.history["loss"]), "| backend:", keras.config.backend())
plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="val"); plt.legend(); plt.title("Keras loss (MSE)"); plt.show()
evaluate("Keras MLP", k_model.predict(Xs_te, verbose=0), secs=keras_time)

## 9. Model 3 – PyTorch + fastai

In [ ]:
from fastai.tabular.all import *
set_seed()
trva = pd.concat([train, val])
splits = (list(range(len(train))), list(range(len(train), len(trva))))
to = TabularPandas(trva.reset_index(drop=True), procs=[Normalize], cont_names=feat_cols, y_names="target",
                   splits=splits, y_block=RegressionBlock())
dls = to.dataloaders(bs=128)
learn = tabular_learner(dls, layers=[64, 32], ps=[.3, .2], metrics=rmse, y_range=None)
t0 = time.time()
learn.fit_one_cycle(15, 1e-3, cbs=[SaveModelCallback(monitor="_rmse", fname="fastai_best"), EarlyStoppingCallback(monitor="_rmse", patience=5)])
fastai_time = time.time() - t0
learn.load("fastai_best")
test_dl = learn.dls.test_dl(test.reset_index(drop=True)[feat_cols])
fp, _ = learn.get_preds(dl=test_dl)
evaluate("fastai Tabular", fp.numpy(), secs=fastai_time)

## 10. Model 4 – PyTorch + TabNet
TabNet uses sequential attention to pick features at each decision step, which gives built-in feature-selection masks (explainability).

In [ ]:
import torch
from pytorch_tabnet.tab_model import TabNetRegressor
set_seed()
tabnet = TabNetRegressor(n_d=8, n_a=8, n_steps=3, gamma=1.3, lambda_sparse=1e-3, seed=SEED, verbose=0,
                         optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=2e-2),
                         scheduler_fn=torch.optim.lr_scheduler.StepLR, scheduler_params=dict(step_size=10, gamma=.7))
t0 = time.time()
tabnet.fit(Xs_tr, y_tr32.reshape(-1, 1), eval_set=[(Xs_va, y_va32.reshape(-1, 1))], eval_metric=["rmse"],
           max_epochs=100, patience=15, batch_size=256, virtual_batch_size=64)
tabnet_time = time.time() - t0
print("Best epoch:", tabnet.best_epoch, "| best val RMSE: %.4f" % tabnet.best_cost)
evaluate("TabNet", tabnet.predict(Xs_te), secs=tabnet_time)

## 11. Model 5 – PyTorch Lightning
*Lightning Flash* (the high-level tabular wrapper named in the brief) is **deprecated and no longer maintained**, and does not install against current PyTorch. The same idea – a structured, reusable training loop with callbacks – is implemented with **PyTorch Lightning** directly.

In [ ]:
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

class LitMLP(L.LightningModule):
    def __init__(self, n_in, lr=1e-3, wd=1e-4):
        super().__init__(); self.save_hyperparameters()
        self.net = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(), nn.Dropout(.3),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Dropout(.2), nn.Linear(32, 1))
    def forward(self, x): return self.net(x).squeeze(-1)
    def training_step(self, b, i):
        x, y = b; loss = nn.functional.mse_loss(self(x), y); self.log("train_loss", loss, prog_bar=False); return loss
    def validation_step(self, b, i):
        x, y = b; self.log("val_loss", nn.functional.mse_loss(self(x), y), prog_bar=False)
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.wd)

set_seed(); L.seed_everything(SEED, workers=True)
mk = lambda X, y, sh: DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)), batch_size=64, shuffle=sh)
tr_dl, va_dl = mk(Xs_tr, y_tr32, True), mk(Xs_va, y_va32, False)
lit = LitMLP(Xs_tr.shape[1])
ckpt = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
trainer = L.Trainer(max_epochs=100, accelerator="cpu", callbacks=[EarlyStopping("val_loss", patience=10), ckpt],
                    enable_progress_bar=False, enable_model_summary=False, logger=False, deterministic=True)
t0 = time.time(); trainer.fit(lit, tr_dl, va_dl); lit_time = time.time() - t0
lit = LitMLP.load_from_checkpoint(ckpt.best_model_path); lit.eval()
with torch.no_grad(): lp = lit(torch.tensor(Xs_te)).numpy()
print("Stopped at epoch:", trainer.current_epoch)
evaluate("PyTorch Lightning", lp, secs=lit_time)

## 12. Robustness check – walk-forward validation (XGBoost)
A single test split can be lucky or unlucky. Here the model is **re-trained every calendar year** on all data before that year (expanding window) and predicts that year, so every test-period prediction is truly out-of-sample.

In [ ]:
wf_rows, wf_pred, wf_true = [], [], []
test_years = sorted(set(test.index.year))
for yr_ in test_years:
    tr_ = data[data.index.year < yr_]; te_ = data[data.index.year == yr_]
    m = xgb.XGBRegressor(n_estimators=max(best.best_iteration, 50), subsample=.8, colsample_bytree=.8, reg_lambda=5,
                         random_state=SEED, n_jobs=-1, **best_cfg).fit(tr_[feat_cols], tr_["target"])
    p = m.predict(te_[feat_cols]); wf_pred += list(p); wf_true += list(te_["target"])
    wf_rows.append(dict(year=yr_, n=len(te_), RMSE=np.sqrt(mean_squared_error(te_["target"], p)),
                        RMSE_zero=np.sqrt(np.mean(te_["target"] ** 2)), DirAcc=np.mean(np.sign(p) == np.sign(te_["target"])) * 100))
wf = pd.DataFrame(wf_rows).round(3); display(wf)
wf_pred, wf_true = np.array(wf_pred), np.array(wf_true)
print("Walk-forward overall  RMSE=%.4f (zero-forecast %.4f) | R2=%.4f | DirAcc=%.1f%%" %
      (np.sqrt(mean_squared_error(wf_true, wf_pred)), np.sqrt(np.mean(wf_true**2)), r2_score(wf_true, wf_pred), np.mean(np.sign(wf_pred)==np.sign(wf_true))*100))

## 13. Model comparison

In [ ]:
# Ensemble (average of the five ML models) as an extra, 'innovation' entry
ml = ["XGBoost", "Keras MLP", "fastai Tabular", "TabNet", "PyTorch Lightning"]
evaluate("Ensemble (avg of 5)", np.mean([preds[m] for m in ml], axis=0), secs=sum(results[m]["Train_s"] for m in ml))
comp = pd.DataFrame(results).T
comp.index.name = "Model"
display(comp.sort_values("RMSE").round(4).style.background_gradient(subset=["RMSE", "MAE"], cmap="RdYlGn_r")
        .background_gradient(subset=["R2", "DirAcc", "Sharpe"], cmap="RdYlGn"))

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4.2))
order = comp.sort_values("RMSE").index
for a, col in zip(ax, ["RMSE", "MAE", "R2", "DirAcc"]):
    comp.loc[order, col].plot.barh(ax=a, color=["tab:gray" if i.startswith("Baseline") else "tab:blue" for i in order]); a.set_title(col); a.invert_yaxis()
ax[3].axvline(50, c="r", ls="--"); ax[3].set_xlim(45, 60)
plt.tight_layout(); plt.show()

In [ ]:
best_name = comp.drop(index=[i for i in comp.index if i.startswith("Baseline")]).sort_values("RMSE").index[0]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].scatter(y_te, preds[best_name], s=6, alpha=.5); ax[0].axhline(0, c="k", lw=.5); ax[0].axvline(0, c="k", lw=.5)
ax[0].set_xlabel("actual next-day return (%)"); ax[0].set_ylabel("predicted (%)"); ax[0].set_title(f"{best_name}: predicted vs actual")
ax[1].plot(test.index, y_te, lw=.5, label="actual"); ax[1].plot(test.index, preds[best_name], lw=.9, label="predicted"); ax[1].legend(); ax[1].set_title("Test period")
plt.tight_layout(); plt.show()
print("Predictions are much smaller in magnitude than actual returns - models shrink toward the mean because the signal-to-noise ratio is very low.")

### Simplicity · Transparency · Efficacy
Scores below combine **measured** values (training time, test RMSE vs the zero baseline) with the usual characteristics of each framework. Edit the 1–5 ratings to reflect *your own* experience when you wrote the code.

In [ ]:
rank = comp.loc[ml, "RMSE"].rank().astype(int)
rubric = pd.DataFrame({
    "Training time (s)": comp.loc[ml, "Train_s"].round(1),
    "Test RMSE": comp.loc[ml, "RMSE"].round(4),
    "Efficacy rank (1=best RMSE)": rank,
    "Simplicity (1-5)":   {"XGBoost": 5, "Keras MLP": 4, "fastai Tabular": 4, "TabNet": 3, "PyTorch Lightning": 2},
    "Transparency (1-5)": {"XGBoost": 4, "Keras MLP": 2, "fastai Tabular": 2, "TabNet": 4, "PyTorch Lightning": 2},
    "Why": {"XGBoost": "few lines, tuned by early stopping; gain/permutation importances, trees are inspectable",
            "Keras MLP": "simple API but black-box weights",
            "fastai Tabular": "very concise pipeline (TabularPandas + learner) but black-box",
            "TabNet": "more hyper-parameters; attention masks give feature importance",
            "PyTorch Lightning": "most boilerplate (module, loaders, trainer) but full control; black-box MLP"}})
display(rubric)

## 14. Explainability – what drives the predictions?

In [ ]:
from sklearn.inspection import permutation_importance
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
imp = pd.Series(best.feature_importances_, index=feat_cols).sort_values()
imp.tail(15).plot.barh(ax=ax[0]); ax[0].set_title("XGBoost – gain importance (top 15)")
pi = permutation_importance(best, X_te, y_te, n_repeats=10, random_state=SEED, scoring="neg_root_mean_squared_error")
pd.Series(pi.importances_mean, index=feat_cols).sort_values().tail(15).plot.barh(ax=ax[1], xerr=pd.Series(pi.importances_std, index=feat_cols)[pd.Series(pi.importances_mean, index=feat_cols).sort_values().tail(15).index])
ax[1].set_title("XGBoost – permutation importance on test\n(RMSE increase when shuffled)")
pd.Series(tabnet.feature_importances_, index=feat_cols).sort_values().tail(15).plot.barh(ax=ax[2]); ax[2].set_title("TabNet – attention importance (top 15)")
plt.tight_layout(); plt.show()

## 15. Economic back-test
Does a statistical edge (if any) translate into money? Each model trades the **test period** long/short based on the sign of its prediction (2 bps cost per position change), compared with **buy-and-hold**. This is a research illustration, not trading advice.

In [ ]:
bt = pd.DataFrame(index=test.index)
bt["Buy & Hold"] = y_te / 100
for m in ml + ["Ensemble (avg of 5)", "Baseline: Ridge"]:
    bt[m] = backtest(preds[m].copy(), y_te)
curve = (1 + bt).cumprod()
fig, ax = plt.subplots(figsize=(12, 5)); curve.plot(ax=ax, lw=1.2); ax.set_title("Cumulative growth of 1 unit (test period)"); ax.set_ylabel("x initial"); plt.show()

def maxdd(x): c = (1 + x).cumprod(); return ((c / c.cummax()) - 1).min() * 100
summary = pd.DataFrame({"Ann. return %": ((1 + bt).prod() ** (252 / len(bt)) - 1) * 100,
                        "Sharpe": bt.apply(sharpe), "Max drawdown %": bt.apply(maxdd)}).round(2)
display(summary.sort_values("Sharpe", ascending=False))

## 16. Conclusion

In [ ]:
non_base = comp.drop(index=[i for i in comp.index if i.startswith("Baseline")])
bm, zb = non_base["RMSE"].idxmin(), comp.loc["Baseline: zero", "RMSE"]
print(f"Best ML model by RMSE      : {bm}  (RMSE {comp.loc[bm,'RMSE']:.4f}, R2 {comp.loc[bm,'R2']:.4f}, DirAcc {comp.loc[bm,'DirAcc']:.1f}%)")
print(f"Zero-return baseline RMSE  : {zb:.4f}")
print(f"Models beating zero baseline on RMSE: {[m for m in non_base.index if comp.loc[m,'RMSE'] < zb] or 'none'}")
print(f"Models with positive R2    : {[m for m in non_base.index if comp.loc[m,'R2'] > 0] or 'none'}")
print(f"Models with Sharpe > buy&hold ({bt['Buy & Hold'].pipe(sharpe):.2f}): {[m for m in non_base.index if summary.loc[m,'Sharpe'] > summary.loc['Buy & Hold','Sharpe']] or 'none'}")

### Key points to carry into the report / viva
* **Task difficulty:** next-day index returns are close to a random walk (EDA: near-zero autocorrelation, fat tails). Expect R² ≈ 0 and directional accuracy only slightly above 50 %; this is a property of the market (weak-form efficiency), not a bug.
* **What *is* predictable:** volatility (strong autocorrelation of squared returns). Volatility/range features dominate the importances; a natural extension is to predict **volatility or direction-with-confidence** instead of the raw return.
* **Framework comparison:** XGBoost is the simplest, fastest and most transparent; Keras and fastai are concise but opaque; TabNet adds interpretability at the cost of tuning effort; Lightning offers control at the cost of boilerplate. Differences in accuracy between the neural frameworks are small because they fit essentially the same function to the same noisy target.
* **Validation discipline:** chronological split, scaler fitted on train only, features built from past data only, and a walk-forward re-test, all to avoid look-ahead bias.
* **Limitations / future work:** add India VIX, USD/INR, FII flows, constituent-bank (HDFC Bank, ICICI, SBI…) returns and news sentiment; classification of up/down with probability thresholds; GARCH benchmark; transaction-cost sensitivity; hyper-parameter search with Optuna.

> The numbers quoted in the report should be taken from the tables above for *your* final run.